# Restaurant Analytics Agent - Tool Definitions

Convert the restaurant data pipeline into reusable tools for the agent. These tools allow the agent to answer both structured questions about ratings and review counts and qualitative questions about customer experiences.

1. **Business Lookup**: Retrieve rating, review count, rating distribution, and restaurant metadata.
2. **Competitor Comparison**: Compare restaurants by rating, review volume, and rating distribution.
3. **Theme Retrieval**: Use vector search over review text to retrieve reviews related to themes like service, food quality, ambience, price, wait time, or complaints.

---
## 1. Configure Source Tables

In [0]:
from pyspark.sql import functions as F

# Source tables
RESTAURANT_METRICS_TABLE = f"{CATALOG}.{SCHEMA}.restaurant_metrics"
RESTAURANTS_TABLE = f"{CATALOG}.{SCHEMA}.restaurants"
REVIEWS_WITH_TEXT_TABLE = f"{CATALOG}.{SCHEMA}.reviews_with_text"
REVIEWS_FOR_VECTOR_TABLE = f"{CATALOG}.{SCHEMA}.reviews_for_vector_search"

print("=" * 60)
print("Configuring agent tool source tables...")
print("=" * 60)

print(f"Restaurant metrics table: {RESTAURANT_METRICS_TABLE}")
print(f"Restaurants table: {RESTAURANTS_TABLE}")
print(f"Review text table: {REVIEWS_WITH_TEXT_TABLE}")
print(f"Vector search source table: {REVIEWS_FOR_VECTOR_TABLE}")

---
## 2. Business Lookup Tool

Objective: Retrieve metrics for a specific restaurant

In [0]:
def business_lookup(restaurant_name: str, limit: int = 5):
    """
    Retrieve rating, review count, rating distribution, and restaurant metadata
    for a specific restaurant.
    """
    
    restaurant_metrics_df = spark.table(RESTAURANT_METRICS_TABLE)
    
    results_df = (
        restaurant_metrics_df
        .filter(F.lower(F.col("business_name")).contains(restaurant_name.lower()))
        .select(
            "gmap_id",
            "business_name",
            "address",
            "avg_rating",
            "review_count",
            "rating_1_count",
            "rating_2_count",
            "rating_3_count",
            "rating_4_count",
            "rating_5_count"
        )
        .orderBy(F.desc("review_count"))
        .limit(limit)
    )
    
    results = results_df.toPandas().to_dict(orient="records")
    
    if len(results) == 0:
        return {
            "status": "not_found",
            "message": f"No restaurant found matching '{restaurant_name}'."
        }
    
    return {
        "status": "success",
        "results": results
    }

---
## 3. Competitor Comparison Tool

Objective: Compare restaurant performance to another restaurant

In [0]:
def competitor_comparison(restaurant_1: str, restaurant_2: str, limit_per_name: int = 3):
    """
    Compare two specific restaurants by rating, review volume, and rating distribution.
    The user provides both restaurant names.
    """
    
    restaurant_metrics_df = spark.table(RESTAURANT_METRICS_TABLE)
    
    restaurant_names = [restaurant_1, restaurant_2]
    all_results = []
    
    for restaurant_name in restaurant_names:
        matches_df = (
            restaurant_metrics_df
            .filter(F.lower(F.col("business_name")).contains(restaurant_name.lower()))
            .select(
                "gmap_id",
                "business_name",
                "address",
                "avg_rating",
                "review_count",
                "rating_1_count",
                "rating_2_count",
                "rating_3_count",
                "rating_4_count",
                "rating_5_count"
            )
            .orderBy(F.desc("review_count"))
            .limit(limit_per_name)
        )
        
        matches = matches_df.toPandas().to_dict(orient="records")
        
        for match in matches:
            match["search_name"] = restaurant_name
        
        all_results.extend(matches)
    
    if len(all_results) == 0:
        return {
            "status": "not_found",
            "message": f"No matching restaurants found for '{restaurant_1}' or '{restaurant_2}'."
        }
    
    return {
        "status": "success",
        "restaurant_1": restaurant_1,
        "restaurant_2": restaurant_2,
        "results": all_results
    }

---
### 3.1. Prepare Review Text Table for Vector Search

Vector Search needs a stable primary key. This creates a new table with a `review_id`

To keep vector search efficient while making the agent more generalizable, this section indexes all available review text for the two selected comparison restaurants and a random sample of 20 additional San Diego Italian restaurants. This allows the agent to answer the primary comparison question while still supporting theme retrieval for other restaurants in the dataset.

In [0]:
# Create a vector-search-ready review text table with a stable primary key
# Include the two selected restaurants plus a random sample of other restaurants

SELECTED_RESTAURANTS = ["cesarina", "parma"]
NUM_RANDOM_RESTAURANTS = 20
MAX_REVIEWS_PER_RANDOM_RESTAURANT = 50

reviews_with_text_df = spark.table(REVIEWS_WITH_TEXT_TABLE)
restaurants_df = spark.table(RESTAURANTS_TABLE)

reviews_joined_df = (
    reviews_with_text_df
    .join(
        restaurants_df.select("gmap_id", "business_name", "address"),
        on="gmap_id",
        how="inner"
    )
    .filter(F.col("text").isNotNull())
    .filter(F.length(F.col("text")) > 10)
)

# Keep all reviews for selected restaurants
selected_reviews_df = (
    reviews_joined_df
    .filter(
        (F.lower(F.col("business_name")).contains("cesarina")) |
        (F.lower(F.col("business_name")).contains("parma"))
    )
)

# Randomly choose additional restaurants, excluding selected restaurants
random_restaurant_ids_df = (
    reviews_joined_df
    .filter(
        ~(
            (F.lower(F.col("business_name")).contains("cesarina")) |
            (F.lower(F.col("business_name")).contains("parma"))
        )
    )
    .select("gmap_id", "business_name")
    .distinct()
    .orderBy(F.rand(seed=42))
    .limit(NUM_RANDOM_RESTAURANTS)
)

# For those random restaurants, take up to N reviews per restaurant
random_reviews_base_df = (
    reviews_joined_df
    .join(
        random_restaurant_ids_df.select("gmap_id"),
        on="gmap_id",
        how="inner"
    )
)

window_by_restaurant = Window.partitionBy("gmap_id").orderBy(F.rand(seed=42))

random_reviews_df = (
    random_reviews_base_df
    .withColumn("random_review_rank", F.row_number().over(window_by_restaurant))
    .filter(F.col("random_review_rank") <= MAX_REVIEWS_PER_RANDOM_RESTAURANT)
    .drop("random_review_rank")
)

# Combine selected restaurants and random sample
reviews_for_vector_df = (
    selected_reviews_df
    .unionByName(random_reviews_df)
    .select(
        F.sha2(
            F.concat_ws(
                "||",
                F.col("gmap_id"),
                F.coalesce(F.col("user_id"), F.lit("unknown_user")),
                F.coalesce(F.col("time").cast("string"), F.lit("unknown_time")),
                F.coalesce(F.col("text"), F.lit(""))
            ),
            256
        ).alias("review_id"),
        "gmap_id",
        "business_name",
        "address",
        "rating",
        "time",
        "text"
    )
    .dropDuplicates(["review_id"])
)

(
    reviews_for_vector_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(REVIEWS_FOR_VECTOR_TABLE)
)

# Change Data Feed is required for Delta Sync Vector Search indexes
spark.sql(f"""
ALTER TABLE {REVIEWS_FOR_VECTOR_TABLE}
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print(f"Created vector-search source table: {REVIEWS_FOR_VECTOR_TABLE}")
print(f"Total rows: {spark.table(REVIEWS_FOR_VECTOR_TABLE).count():,}")

display(
    spark.table(REVIEWS_FOR_VECTOR_TABLE)
    .groupBy("business_name")
    .agg(F.count("*").alias("review_text_count"))
    .orderBy(F.desc("review_text_count"))
)

---
### 3.2. Initialize Vector Search Client

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

print("Vector Search client initialized.")

---
### 3.3. Configure Vector Search Endpoint and Index

In [0]:
# Vector Search configuration
VECTOR_SEARCH_ENDPOINT_NAME = "restaurant_reviews_vs_endpoint"
VECTOR_SEARCH_INDEX_NAME = f"{CATALOG}.{SCHEMA}.restaurant_review_text_index"

# Databricks embedding model endpoint
# If this endpoint is not available in your workspace, replace it with the embedding endpoint your course/workspace provides.
EMBEDDING_MODEL_ENDPOINT_NAME = "databricks-gte-large-en"

print("=" * 60)
print("Configuring Vector Search index...")
print("=" * 60)

print(f"Endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")
print(f"Index: {VECTOR_SEARCH_INDEX_NAME}")
print(f"Source table: {REVIEWS_FOR_VECTOR_TABLE}")
print(f"Embedding endpoint: {EMBEDDING_MODEL_ENDPOINT_NAME}")

---
### 3.4. Create or Connect to Vector Search Endpoint

Create an endpoint/index if needed, otherwise connect to the existing one

In [0]:
# Create Vector Search endpoint if needed

try:
    vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✓ Found existing endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")

except Exception:
    print(f"Creating endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")
    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )
    vsc.wait_for_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✓ Created endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")

---
### 3.5. Create or Connect to Vector Search Index

In [0]:
# Delete offline / partial Vector Search index

try:
    print(f"Deleting index: {VECTOR_SEARCH_INDEX_NAME}")
    
    vsc.delete_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    
    print(f"Deleted index: {VECTOR_SEARCH_INDEX_NAME}")

except Exception as e:
    print("Index delete failed or index was already gone.")
    print(e)

In [0]:
# Create Delta Sync index if needed

try:
    review_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    print(f"✓ Found existing index: {VECTOR_SEARCH_INDEX_NAME}")

except Exception:
    print(f"Creating index: {VECTOR_SEARCH_INDEX_NAME}")
    
    review_index = vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME,
        source_table_name=REVIEWS_FOR_VECTOR_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="review_id",
        embedding_source_column="text",
        embedding_model_endpoint_name=EMBEDDING_MODEL_ENDPOINT_NAME,
        columns_to_sync=[
            "review_id",
            "gmap_id",
            "business_name",
            "address",
            "rating",
            "time",
            "text"
        ]
    )
    
    review_index.wait_until_ready()
    print(f"✓ Created index: {VECTOR_SEARCH_INDEX_NAME}")

---
### 3.6. Sync Vector Search Index

In [0]:
# Sync the index after creating or updating the source table

review_index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=VECTOR_SEARCH_INDEX_NAME
)

review_index.sync()
review_index.wait_until_ready()

print("✓ Vector Search index is synced and ready.")

---
### 3.7. Helper Function to Parse Vector Search Results

In [0]:
def parse_vector_search_results(results):
    """
    Convert Databricks Vector Search response into a list of dictionaries.
    """
    
    columns = [col["name"] for col in results["manifest"]["columns"]]
    rows = results["result"]["data_array"]
    
    parsed_results = []
    
    for row in rows:
        parsed_results.append(dict(zip(columns, row)))
    
    return parsed_results

---
## 4. Tool 3 - Theme Retrieval

In [0]:
def theme_retrieval(restaurant_name: str, theme_query: str, num_results: int = 10):
    """
    Retrieve reviews related to a user-provided theme using Databricks Vector Search.
    This uses semantic similarity search over review text and does not rely on
    hard-coded keywords.
    """
    
    review_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    
    # Pull more than needed because restaurant filtering happens after retrieval.
    raw_num_results = max(num_results * 5, 25)
    
    results = review_index.similarity_search(
        query_text=theme_query,
        columns=[
            "review_id",
            "business_name",
            "address",
            "rating",
            "time",
            "text"
        ],
        num_results=raw_num_results
    )
    
    parsed_results = parse_vector_search_results(results)
    
    filtered_results = [
        row for row in parsed_results
        if restaurant_name.lower() in row.get("business_name", "").lower()
    ]
    
    filtered_results = filtered_results[:num_results]
    
    if len(filtered_results) == 0:
        return {
            "status": "not_found",
            "message": (
                f"No vector search results found for restaurant '{restaurant_name}' "
                f"and theme query '{theme_query}'."
            ),
            "restaurant_name": restaurant_name,
            "theme_query": theme_query,
            "results": []
        }
    
    return {
        "status": "success",
        "restaurant_name": restaurant_name,
        "theme_query": theme_query,
        "results": filtered_results
    }

### 6.12 Validate All Three Tools

In [0]:
print("=" * 60)
print("Validating agent tools...")
print("=" * 60)

print("\nTool 1: Business Lookup")
business_lookup_result = business_lookup("Cesarina")

if business_lookup_result["status"] == "success":
    display(business_lookup_result["results"])
else:
    print(business_lookup_result["message"])


print("\nTool 2: Competitor Comparison")
comparison_result = competitor_comparison("Cesarina", "Parma Cucina Italiana")

if comparison_result["status"] == "success":
    display(comparison_result["results"])
else:
    print(comparison_result["message"])


print("\nTool 3: Theme Retrieval")
try:
    theme_result = theme_retrieval(
        "Cesarina",
        "reviews about pasta quality and authentic Italian food",
        num_results=5
    )
    
    if theme_result["status"] == "success":
        display(theme_result["results"])
    else:
        print(theme_result["message"])

except Exception as e:
    print("Theme Retrieval failed.")
    print(e)